# EXERCÍCIO 02 — Detector simples de alucinação (Ollama)

🎯 O que esse notebook vai fazer

Ele vai:

- Perguntar algo para o modelo (Ollama)
- Receber a resposta
- Comparar com uma mini base confiável local
- Classificar:
  - ✔️ correto
  - ⚠️ suspeito
  - ❌ alucinado

🧠 Estratégia (simples e eficaz)

Vamos usar um método bem direto:

📚 “Base de verdade local”

Um dicionário Python com fatos corretos.

🔍 Comparação por palavras-chave

Se a resposta:

- contém fatos da base → ✔️
- mistura coisa certa + errada → ⚠️
- inventa coisas fora da base → ❌

In [ ]:
import requests

# =========================
# CONFIG OLLAMA
# =========================
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.1:8b"

# =========================
# FUNÇÃO: chamar LLM
# =========================
def ask_llm(prompt):
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(OLLAMA_URL, json=payload)
    return response.json()["response"]

# =========================
# BASE DE VERDADE (mini KB)
# =========================
TRUTH_BASE = {
    "docker": [
        "container",
        "isolamento",
        "imagem",
        "ambiente"
    ],
    "kubernetes": [
        "orquestração",
        "containers",
        "escalabilidade",
        "pods"
    ],
    "langgraph": [
        "langchain",
        "grafo",
        "agentes",
        "workflow"
    ]
}

# =========================
# FUNÇÃO: detecção simples
# =========================
def detect_hallucination(topic, answer):
    answer_lower = answer.lower()

    if topic not in TRUTH_BASE:
        return "⚠️ tópico desconhecido na base"

    keywords = TRUTH_BASE[topic]

    matches = 0
    for kw in keywords:
        if kw in answer_lower:
            matches += 1

    score = matches / len(keywords)

    print("\n🔎 SCORE:", score)

    if score >= 0.6:
        return "✔️ correto (provavelmente alinhado)"
    elif score >= 0.3:
        return "⚠️ suspeito (mistura de certo e errado)"
    else:
        return "❌ provável alucinação"

# =========================
# TESTE DO LAB
# =========================

topic = "langgraph"

prompt = f"""
Explique o que é {topic} de forma técnica e objetiva.
"""

print("🧠 Perguntando ao modelo...\n")

response = ask_llm(prompt)

print("\n===== RESPOSTA DO MODELO =====\n")
print(response)

result = detect_hallucination(topic, response)

print("\n===== CLASSIFICAÇÃO =====\n")
print(result)